
**How would we execute a query like this?**
``` sql
SELECT * 
FROM persons JOIN books
     ON name = author
WHERE price < 10
```




**Create some fake data**

In [ ]:
from system.data_classes import *
from faker import Faker


Faker.seed(42)
fake = Faker()

number_of_tuples = 10000

# create some fake data mimicking tuples in a relation
persons = [Person(fake.name(), fake.date()) for _ in range(number_of_tuples)]
books = [
    Book(
        fake.sentence(),
        persons[i % number_of_tuples].name,
        fake.pyfloat(min_value=0, max_value=100, right_digits=2),
    )
    for i in range(number_of_tuples)
]
persons[:10], books[:10]

**Emulate store**

In [ ]:
import pickle


# write data into file to emulate store
with open("persons.pkl", "wb") as f:
    pickle.dump([vars(person) for person in persons], f) # convert into dictionary (aka Tuple type)
with open("books.pkl", "wb") as f:
    pickle.dump([vars(book) for book in books], f) # convert into dictionary (aka Tuple type)

**Create QEP**

In [ ]:
from system.operators import *


scan_persons = Scan("persons.pkl")
scan_books = Scan("books.pkl")
filter_books = Filter(scan_books, "price < 10")
join = SHJ(scan_persons, filter_books, "name", "author")
query_plan = Print(join)

query_plan.dump()

**Interpret QEP**

In [ ]:
query_plan.interpret_open()

**Compile QEP**

In [ ]:
code = query_plan.compile()
print(code)

**Compile generated Python code to byte code and run executable**

In [ ]:
from py_compile import compile
import subprocess


# write generated code into file
with open("codegen.py", "w") as f:
    f.write(code)

# create executable file
with open("codegen.pyc", "w"):
    pass

# compile Python code to byte code
compile("codegen.py", "codegen.pyc", doraise=True)
# run executable
subprocess.call(["python3", "codegen.pyc"])

**But how fast is each execution paradigm?**

In [ ]:
from time import time_ns


num_runs = 5
scale_factors = [0.1, 0.25, 0.5, 0.75, 1]


def measure_interpreter(query_plan: Operator) -> list[int]:
    """
    Measures the running time in ms using the interpreter.
    :param query_plan: QEP to execute.
    :return: List of `num_runs` measurements.
    """

    res = list()
    for i in range(num_runs):
        interpret_start = time_ns()
        query_plan.interpret_open()
        interpret_end = time_ns()
        res.append((interpret_end - interpret_start) / 1000)
    return res


def measure_compiler(query_plan: Operator) -> list[int]:
    """
    Measures the running time in ms using the compiler.
    :param query_plan: QEP to execute.
    :return: List of `num_runs` measurements.
    """

    res = list()
    for i in range(num_runs):
        compile_start = time_ns()
        code = query_plan.compile()
        with open("codegen.py", "w") as f:
            f.write(code)
        with open("codegen.pyc", "w"):
            pass
        compile("codegen.py", "codegen.pyc", doraise=True)
        subprocess.call(["python3", "codegen.pyc"])
        compile_end = time_ns()
        res.append((compile_end - compile_start) / 1000)
    return res


def create_qep(sf: int):
    scan_persons = Scan("persons.pkl", number_of_tuples * sf)
    scan_books = Scan("books.pkl", number_of_tuples * sf)
    filter_books = Filter(scan_books, "price < 10")
    join = SHJ(scan_persons, filter_books, "name", "author")
    return Count(join)


data: list[tuple[str, int, int]] = list()
for sf in scale_factors:
    query_plan = create_qep(sf)
    for time in measure_interpreter(query_plan):
        data.append(("interpreter", sf, time))
    for time in measure_compiler(query_plan):
        data.append(("compiler", sf, time))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns


df = pd.DataFrame(data, columns=["approach", "scale_factor", "running_time"])
ax = sns.barplot(
    df, x="scale_factor", y="running_time", hue="approach", estimator=np.median
)
ax.set_xlabel("Scale Factor")
ax.set_ylabel("Running time (ms)")